In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms
import torch.optim as optim
import numpy as np
import pandas as pd 
import json

from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_score, recall_score, f1_score
from medmnist import INFO

from tqdm.notebook import tqdm

from medmnist import ChestMNIST

In [2]:
import ctypes

cuda = ctypes.CDLL("/run/opengl-driver/lib/libcuda.so.1")
print("CUDA initialization:", cuda.cuInit(0))


CUDA initialization: 0


In [3]:
torch.cuda.get_device_name()

'NVIDIA GeForce RTX 3050 Laptop GPU'

In [4]:
device = torch.device("cuda")
device

device(type='cuda')

In [5]:
Loss = nn.BCEWithLogitsLoss() #over crossEntropy BCE lets each label work with it's own signmoid

In [6]:
trans = transforms.Compose([transforms.ToTensor(),
                            transforms.Normalize(mean=[0.5], std=[0.5])])

In [7]:
train_df = ChestMNIST(split="train", transform=trans ,download=True)
val_df = ChestMNIST(split="val", transform=trans, download=True)
test_df = ChestMNIST(split="test", transform=trans, download=True)

In [8]:
batch_size = 64 

train_loader = DataLoader(train_df, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_df, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_df, batch_size=batch_size, shuffle=False)

In [9]:
images, labels = next(iter(train_loader))

f"{images.shape}, {labels.shape}, {images.dtype}, {labels.dtype}"

'torch.Size([64, 1, 28, 28]), torch.Size([64, 14]), torch.float32, torch.int64'

In [10]:
from torchvision.models import resnet18
model = resnet18(weights=None)
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [11]:
model.conv1 = nn.Conv2d(in_channels=1, out_channels=64, kernel_size=7, stride=2, padding=3, bias=False)
model.fc = nn.Linear(in_features=model.fc.in_features, out_features=14)

In [12]:
model = model.to(device)

In [13]:
images, labels = next(iter(train_loader))

images = images.to(device)

In [14]:
with torch.no_grad():
    op = model(images)

f"{images.shape}, {op.shape}"

'torch.Size([64, 1, 28, 28]), torch.Size([64, 14])'

In [15]:
optim = optim.Adam(model.parameters(), lr=0.01)

In [16]:
def epoch0(model, train_loader, Loss, optim, device):
    model.train()
    running_loss = 0.0

    for images, labels in tqdm(train_loader, desc="Training", leave=False):

        images = images.to(device)
        labels = labels.float().to(device)

        optim.zero_grad()

        ops = model(images)

        loss = Loss(ops, labels)

        loss.backward()

        optim.step()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(train_loader.dataset)

In [17]:
@torch.no_grad()
def val0(model, val_loader, Loss, device):
    model.eval()
    running_loss = 0.0

    for images, labels in tqdm(val_loader, desc="Validating", leave=False):

        images = images.to(device)
        labels = labels.float().to(device)

        ops = model(images)

        loss = Loss(ops, labels)

        running_loss += loss.item() * images.size(0)
    
    return running_loss / len(val_loader.dataset)

In [18]:
epochs = 10 

train_losses = []
val_losses = []

for epoch in range(epochs):

    train_loss = epoch0(model, train_loader, Loss, optim, device)
    val_loss = val0(model, val_loader, Loss, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch: {epoch+1}, Train loss: {train_loss}, val loss: {val_loss}")

Training:   0%|          | 0/1227 [00:00<?, ?it/s]

Validating:   0%|          | 0/176 [00:00<?, ?it/s]

Epoch: 1, Train loss: 0.18049870981718658, val loss: 0.1746104407774296


Training:   0%|          | 0/1227 [00:00<?, ?it/s]

Validating:   0%|          | 0/176 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/176 [00:00<?, ?it/s]

Epoch: 9, Train loss: 0.16328441833963014, val loss: 0.16342230099457153


Training:   0%|          | 0/1227 [00:00<?, ?it/s]

Validating:   0%|          | 0/176 [00:00<?, ?it/s]

Epoch: 10, Train loss: 0.16199108609395962, val loss: 0.16271879853321616


In [19]:
model.eval()

all_probs = []
all_labels = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing rn"):
        images = images.to(device)

        logits = model(images)
        probs = torch.sigmoid(logits)

        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.numpy())

y_prob = np.concatenate(all_probs, axis=0)
y_true = np.concatenate(all_labels, axis=0)

f"Prob shape: {y_prob.shape}, Labels shape: {y_true.shape}"

Testing rn:   0%|          | 0/351 [00:00<?, ?it/s]

'Prob shape: (22433, 14), Labels shape: (22433, 14)'

In [20]:
auc_score = []

for i in range(14):
    auc = roc_auc_score(y_true[:,i], y_prob[:,i])
    auc_score.append(auc)

f"macro auc score {np.mean(auc_score)}"

'macro auc score 0.7437807569829343'

In [21]:
y_pred = (y_prob >= 0.5).astype(int)

In [22]:
precision = precision_score(y_true, y_pred, average="macro", zero_division=0)
recall = recall_score(y_true, y_pred, average="macro", zero_division=0)
f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)

f"precision: {precision}, recall: {recall}, f1: {f1}"

'precision: 0.242538728956386, recall: 0.007907795450237464, f1: 0.014662628309757968'

In [23]:
thresholds = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5]

In [24]:
rows = []

for threshold in thresholds:
    preds = (y_prob >= threshold).astype(int)
    rows.append({
        "Threshold": threshold, "Precision": precision_score(y_true, preds, average="macro", zero_division=0),
        "Recall": recall_score(y_true, preds, average="macro", zero_division=0),
        "F1": f1_score(y_true, preds, average="macro", zero_division=0)})

results = pd.DataFrame(rows)
results

,Threshold,Precision,Recall,F1
0,0.05,0.102907,0.541359,0.162664
1,0.10,0.126604,0.334402,0.176593
2,0.15,0.143641,0.210961,0.156068
3,0.20,0.150876,0.135488,0.125511
4,0.25,0.171870,0.091771,0.101955
5,0.30,0.172957,0.061852,0.079234
6,0.35,0.206581,0.039836,0.058086
7,0.40,0.227203,0.024859,0.040575
8,0.45,0.179071,0.014582,0.025790
9,0.50,0.242539,0.007908,0.014663


In [25]:
torch.save(model.state_dict(), "resnet18_chestMNIST.pth")

In [26]:
model.eval()
dummy = torch.randn(1, 1, 28, 28).to(device)

torch.onnx.export(  model, (dummy,), "resnet18_chestMNIST.onnx", input_names=["input"], output_names=["logits"],
                    dynamic_axes={"input": {0: "batch_size"}, "logits": {0: "batch_size"}},opset_version=17, dynamo=False   )


/tmp/ipykernel_7515/2845617342.py:4: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(  model, (dummy,), "resnet18_chestMNIST.onnx", input_names=["input"], output_names=["logits"],


In [27]:
class ChestCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
                                        
                                        nn.Conv2d(1, 32, kernel_size=3, padding=1),
                                        nn.BatchNorm2d(32),
                                        nn.ReLU(),
                                        nn.MaxPool2d(2),

                                        
                                        nn.Conv2d(32, 64, kernel_size=3, padding=1),
                                        nn.BatchNorm2d(64),
                                        nn.ReLU(),
                                        nn.MaxPool2d(2),

                                        
                                        nn.Conv2d(64, 128, kernel_size=3, padding=1),
                                        nn.BatchNorm2d(128),
                                        nn.ReLU(),
                                        nn.MaxPool2d(2)
                                                                                        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 14)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [29]:
rn_model = model
cnn_model = ChestCNN().to(device)
model = cnn_model

In [32]:
Loss =  nn.BCEWithLogitsLoss()

optim = torch.optim.Adam(model.parameters(), lr=0.01)

In [33]:
rnn_train_looses = train_losses
rnn_val_losses = val_losses

cnn_train_losses = []
cnn_val_losses = []


In [34]:
epochs = 10

for epoch in range(epochs):

    train_loss = epoch0(model, train_loader, Loss, optim, device)
    val_loss = val0(model, val_loader, Loss, device)

    cnn_train_losses.append(train_loss)
    cnn_val_losses.append(val_loss)

    print(f"Epoch: {epoch+1}, Train loss: {train_loss}, val loss: {val_loss}")

Training:   0%|          | 0/1227 [00:00<?, ?it/s]

Validating:   0%|          | 0/176 [00:00<?, ?it/s]

Epoch: 1, Train loss: 0.1807403520958172, val loss: 0.17007918954851878


Training:   0%|          | 0/1227 [00:00<?, ?it/s]

Validating:   0%|          | 0/176 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/176 [00:00<?, ?it/s]

Epoch: 9, Train loss: 0.16618857037135693, val loss: 0.16433227259729813


Training:   0%|          | 0/1227 [00:00<?, ?it/s]

Validating:   0%|          | 0/176 [00:00<?, ?it/s]

Epoch: 10, Train loss: 0.16577806466154568, val loss: 0.166515176181902
